# Présidentielles françaises 2002-2022 : analyse et tendance 2027

Notebook principal du projet. Il suit le processus KDD :
1. Sélection des données
2. Prétraitement
3. Transformation
4. Fouille (analyse, tendance 2027)
5. Interprétation

Le code est organisé en fonctions, appelées depuis `main()` à la fin du notebook.
Les données brutes doivent être placées dans `data/raw/` (voir le README).

In [ ]:
from pathlib import Path
import json
import re
import unicodedata

import numpy as np

import pandas as pd
import plotly.express as px
import pyarrow.parquet as pq

DOSSIER_PROJET = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DOSSIER_RAW = DOSSIER_PROJET / "data" / "raw"
DOSSIER_REFERENCE = DOSSIER_PROJET / "data" / "reference"
DOSSIER_PROCESSED = DOSSIER_PROJET / "data" / "processed"

ANNEES = [2002, 2007, 2012, 2017, 2022]
ELECTIONS = [f"{annee}_pres_t{tour}" for annee in ANNEES for tour in (1, 2)]

## 1. Sélection des données

In [ ]:
def charger_resultats_generaux(chemin=DOSSIER_RAW / "general_results.parquet", elections=ELECTIONS):
    """Charge les résultats généraux (inscrits, votants, abstention...) des présidentielles, par bureau de vote."""
    table = pq.read_table(chemin, filters=[("id_election", "in", elections)])
    return table.to_pandas()


def charger_resultats_candidats(chemin=DOSSIER_RAW / "candidats_results.parquet", elections=ELECTIONS):
    """Charge les voix par candidat et par bureau de vote pour les présidentielles."""
    colonnes = ["id_election", "code_departement", "code_commune", "code_bv",
                "nom", "prenom", "voix", "ratio_voix_inscrits", "ratio_voix_exprimes"]
    table = pq.read_table(chemin, columns=colonnes, filters=[("id_election", "in", elections)])
    return table.to_pandas()


def charger_familles(chemin=DOSSIER_REFERENCE / "candidats_familles.csv"):
    """Charge la table qui associe chaque candidat à une famille politique."""
    return pd.read_csv(chemin)

## 2. Prétraitement

In [ ]:
def normaliser_nom(nom):
    """Met un nom en majuscules sans accents, pour comparer les années entre elles."""
    sans_accents = unicodedata.normalize("NFKD", str(nom)).encode("ascii", "ignore").decode()
    return sans_accents.upper().strip()


def ajouter_annee_tour(df):
    """Ajoute les colonnes annee et tour à partir de id_election (ex. 2022_pres_t1)."""
    df = df.copy()
    df["annee"] = df["id_election"].str[:4].astype(int)
    df["tour"] = df["id_election"].str[-1].astype(int)
    return df


def ajouter_familles(candidats, familles):
    """Associe chaque ligne de candidat à sa famille politique (jointure sur l'année et le nom)."""
    familles = familles.assign(cle_nom=familles["nom"].map(normaliser_nom))[["annee", "cle_nom", "famille"]]
    candidats = candidats.assign(cle_nom=candidats["nom"].map(normaliser_nom))
    resultat = candidats.merge(familles, on=["annee", "cle_nom"], how="left").drop(columns="cle_nom")
    manquants = resultat.loc[resultat["famille"].isna(), "nom"].unique()
    if len(manquants) > 0:
        print("Candidats sans famille politique :", list(manquants))
    return resultat

## 3. Transformation

In [ ]:
def agreger_par_departement(generaux):
    """Somme les résultats généraux par élection et par département, puis recalcule les taux."""
    colonnes = ["inscrits", "abstentions", "votants", "blancs", "nuls", "exprimes"]
    dep = (generaux.groupby(["id_election", "annee", "tour", "code_departement", "libelle_departement"])[colonnes]
           .sum().reset_index())
    dep["taux_abstention"] = dep["abstentions"] / dep["inscrits"]
    dep["taux_blancs_nuls"] = (dep["blancs"] + dep["nuls"]) / dep["votants"]
    return dep


def voix_par_famille_departement(candidats, generaux_dep):
    """Calcule la part des exprimés obtenue par chaque famille politique, par élection et par département."""
    voix = (candidats.groupby(["id_election", "annee", "tour", "code_departement", "famille"])["voix"]
            .sum().reset_index())
    voix = voix.merge(generaux_dep[["id_election", "code_departement", "exprimes"]],
                      on=["id_election", "code_departement"], how="left")
    voix["part_exprimes"] = voix["voix"] / voix["exprimes"]
    return voix

## Cartes

Les contours des départements de métropole viennent de [france-geojson](https://github.com/gregoiredavid/france-geojson). L'outre-mer et les Français de l'étranger ne sont pas sur la carte.

In [ ]:
def charger_contours(chemin=DOSSIER_REFERENCE / "departements.geojson"):
    """Charge les contours des départements de métropole (GeoJSON, propriété code = code du département)."""
    with open(chemin, encoding="utf-8") as f:
        return json.load(f)


def carte_departements(df, colonne, titre, contours=None):
    """Dessine une carte choroplèthe des départements pour une colonne de df (indexée par code_departement)."""
    contours = contours or charger_contours()
    codes_metropole = {f["properties"]["code"] for f in contours["features"]}
    df = df[df["code_departement"].isin(codes_metropole)]
    fig = px.choropleth(df, geojson=contours, locations="code_departement",
                        featureidkey="properties.code", color=colonne,
                        hover_name="code_departement", color_continuous_scale="Viridis",
                        title=titre)
    fig.update_geos(fitbounds="locations", visible=False)
    fig.update_layout(margin=dict(l=0, r=0, t=40, b=0))
    return fig

## 5. Tendance 2027

Méthode :
1. Pour chaque département et chaque famille politique, on calcule l'**écart** entre le score du département et le score national, au 1er tour de chaque présidentielle (2002 à 2022).
2. On prolonge cet écart jusqu'en 2027 avec une régression linéaire (pente calculée sur les cinq élections).
3. On ajoute cet écart au score national de la famille dans le **sondage de référence** (hypothèse 1, Elabe, 17 octobre 2026).
4. Les scores négatifs sont ramenés à 0, puis chaque département est remis à 100 %.

C'est une **tendance**, pas une prévision : elle suppose que la géographie du vote continue d'évoluer comme entre 2002 et 2022.
On la compare ensuite à la projection publiée avec les sondages (site Repère 2027).

In [ ]:
def lire_csv_sondage(nom_fichier):
    """Lit un CSV des fichiers sondages 2027 (séparateur ; et virgule décimale)."""
    return pd.read_csv(DOSSIER_RAW / nom_fichier, sep=";", decimal=",", encoding="utf-8-sig",
                       dtype={"code_departement": str})


def sondage_reference_hypothese1():
    """Extrait les scores de l'hypothèse 1 du sondage de référence, écrits dans la colonne principaux_enseignements."""
    sondages = lire_csv_sondage("sondages-presidentielle-2027.csv")
    reference = sondages[sondages["vague_de_reference"] == "oui"].iloc[0]
    texte = reference["principaux_enseignements"].split("(hyp. 1")[0]
    scores = re.findall(r"([A-ZÉÈ][\w\-' ]*?) (\d+(?:,\d+)?)(?= ·|$)", texte.strip())
    df = pd.DataFrame(scores, columns=["nom", "intentions_pct"])
    df["intentions_pct"] = df["intentions_pct"].str.replace(",", ".").astype(float)
    df = df[~df["nom"].str.contains("Terrain|personnes|inscrits")]
    df["date_sondage"] = reference["date_publication_iso"]
    df["institut"] = reference["institut"]
    return df.reset_index(drop=True)


def familles_2027(sondage, chemin=DOSSIER_REFERENCE / "candidats_familles_2027.csv"):
    """Associe chaque candidat du sondage 2027 à sa famille politique et somme les intentions par famille."""
    table = pd.read_csv(chemin)
    table["cle_nom"] = table["nom"].map(normaliser_nom)
    sondage = sondage.assign(cle_nom=sondage["nom"].map(normaliser_nom))
    # Le sondage ne donne que le nom de famille : on compare avec la fin du nom de la table (ex. « DE VILLEPIN »).
    sondage["famille"] = sondage["cle_nom"].map(
        lambda n: next((f for c, f in zip(table["cle_nom"], table["famille"]) if c == n or c.endswith(" " + n)), None))
    manquants = sondage.loc[sondage["famille"].isna(), "nom"].tolist()
    if manquants:
        print("Candidats 2027 sans famille :", manquants)
    return sondage, sondage.groupby("famille")["intentions_pct"].sum() / 100


def parts_premier_tour(familles_dep, generaux_dep):
    """Table complète (département x famille x année) des parts au 1er tour, avec 0 quand une famille n'a pas de candidat."""
    t1 = familles_dep[familles_dep["tour"] == 1]
    grille = pd.MultiIndex.from_product([t1["code_departement"].unique(), t1["famille"].unique(), ANNEES],
                                        names=["code_departement", "famille", "annee"])
    parts = t1.set_index(["code_departement", "famille", "annee"])["part_exprimes"].reindex(grille, fill_value=0)
    return parts.reset_index()


def parts_nationales(candidats, generaux):
    """Part nationale de chaque famille au 1er tour de chaque présidentielle."""
    t1 = candidats[candidats["tour"] == 1]
    voix = t1.groupby(["annee", "famille"])["voix"].sum()
    exprimes = generaux[generaux["tour"] == 1].groupby("annee")["exprimes"].sum()
    return (voix / exprimes.reindex(voix.index.get_level_values("annee")).values).rename("part_nationale").reset_index()


def tendance_2027(parts_dep, parts_nat, intentions_famille, annee_cible=2027):
    """Prolonge l'écart département/national de chaque famille et l'applique au sondage 2027."""
    df = parts_dep.merge(parts_nat, on=["annee", "famille"], how="left").fillna({"part_nationale": 0})
    df["ecart"] = df["part_exprimes"] - df["part_nationale"]

    def prolonger(groupe):
        pente, origine = np.polyfit(groupe["annee"], groupe["ecart"], 1)
        dernier = groupe.loc[groupe["annee"].idxmax(), "ecart"]
        return pd.Series({"ecart_2022": dernier, "pente_par_an": pente,
                          "ecart_2027": dernier + pente * (annee_cible - groupe["annee"].max())})

    ecarts = df.groupby(["code_departement", "famille"]).apply(prolonger, include_groups=False).reset_index()
    ecarts["intention_nationale"] = ecarts["famille"].map(intentions_famille).fillna(0)
    ecarts["part_brute"] = (ecarts["intention_nationale"] + ecarts["ecart_2027"]).clip(lower=0)
    ecarts.loc[ecarts["intention_nationale"] == 0, "part_brute"] = 0
    total = ecarts.groupby("code_departement")["part_brute"].transform("sum")
    ecarts["part_2027"] = ecarts["part_brute"] / total
    return ecarts.drop(columns="part_brute")


CANDIDATS_PROJETES = {"MARINE LE PEN": ("LE PEN", "extreme_droite"),
                      "EDOUARD PHILIPPE": ("PHILIPPE", "centre"),
                      "JEAN-LUC MELENCHON": ("MELENCHON", "gauche_radicale")}


def comparer_projection_site(tendance, sondage):
    """Compare notre tendance à la projection départementale du site pour les 3 candidats qu'il projette."""
    site = lire_csv_sondage("projection-departementale-1er-tour.csv")
    infos = site["candidat"].map(normaliser_nom).map(CANDIDATS_PROJETES)
    site["cle_nom"], site["famille"] = infos.str[0], infos.str[1]
    # Poids du candidat dans sa famille (sondage national), pour passer de la famille au candidat.
    sondage = sondage.assign(poids=sondage["intentions_pct"] / sondage.groupby("famille")["intentions_pct"].transform("sum"))
    site = site.merge(sondage[["cle_nom", "poids"]], on="cle_nom", how="left")
    site = site.merge(tendance[["code_departement", "famille", "part_2027"]], on=["code_departement", "famille"], how="left")
    site["notre_estimation_pct"] = 100 * site["part_2027"] * site["poids"]
    resume = site.groupby("candidat").apply(lambda g: pd.Series({
        "correlation": g["score_estime_pct"].corr(g["notre_estimation_pct"]),
        "ecart_moyen_points": (g["notre_estimation_pct"] - g["score_estime_pct"]).abs().mean()}),
        include_groups=False)
    return site, resume

## 4. Contrôles

In [ ]:
def resume_national(candidats):
    """Affiche les quatre premiers candidats au niveau national pour chaque élection (contrôle de cohérence)."""
    totaux = candidats.groupby(["id_election", "nom"])["voix"].sum().reset_index()
    return (totaux.sort_values(["id_election", "voix"], ascending=[True, False])
            .groupby("id_election").head(4))

## Main

In [ ]:
def main():
    generaux = ajouter_annee_tour(charger_resultats_generaux())
    candidats = ajouter_annee_tour(charger_resultats_candidats())
    candidats = ajouter_familles(candidats, charger_familles())

    print(f"Résultats généraux : {len(generaux):,} lignes")
    print(f"Résultats candidats : {len(candidats):,} lignes")

    generaux_dep = agreger_par_departement(generaux)
    familles_dep = voix_par_famille_departement(candidats, generaux_dep)

    DOSSIER_PROCESSED.mkdir(parents=True, exist_ok=True)
    generaux_dep.to_parquet(DOSSIER_PROCESSED / "participation_departements.parquet", index=False)
    familles_dep.to_parquet(DOSSIER_PROCESSED / "familles_departements.parquet", index=False)

    display(resume_national(candidats))

    exemple = familles_dep[(familles_dep["id_election"] == "2022_pres_t1")
                           & (familles_dep["famille"] == "extreme_droite")]
    carte_departements(exemple, "part_exprimes",
                       "Extrême droite au 1er tour 2022 (part des exprimés)").show()
    sondage, intentions_famille = familles_2027(sondage_reference_hypothese1())
    print("Intentions 2027 par famille (hypothèse 1) :")
    display(intentions_famille.sort_values(ascending=False))

    parts_dep = parts_premier_tour(familles_dep, generaux_dep)
    tendance = tendance_2027(parts_dep, parts_nationales(candidats, generaux), intentions_famille)
    tendance.to_parquet(DOSSIER_PROCESSED / "tendance_2027_departements.parquet", index=False)

    site, comparaison = comparer_projection_site(tendance, sondage)
    print("Comparaison avec la projection du site (par département) :")
    display(comparaison)

    ed = tendance[tendance["famille"] == "extreme_droite"]
    carte_departements(ed, "part_2027", "Tendance 2027 : extrême droite au 1er tour (part des exprimés)").show()
    return generaux_dep, familles_dep, tendance


if __name__ == "__main__":
    generaux_dep, familles_dep, tendance = main()